# Heston Model results 
---
This notebook presents the results of the sobolev training on GAN-model pre-trained on the **Heston model**.

---
Google Colab case

If you run this notebook on Google Colab, the utils folder is not available by default.
You must therefore download the utils.zip file provided with this notebook and then run the next cell, which extracts the archive and makes the utils folder accessible to Python.

Local case

If you already have the utils folder in the correct location, or if your environment is properly configured,
you can leave the next cell as is, or simply keep it commented.

In [ ]:
import os

BASE = "/content"
os.chdir(BASE)
print("Working directory:", os.getcwd())

# If utils directory does not exist, unzip it
if not os.path.isdir(os.path.join(BASE, "utils")):
    !unzip "utils.zip" -d /content


In [2]:
import torch
import torch.nn as nn

__dtype__ = torch.float32
__device__ = torch.device('cuda:0')

T_min = 50/365
T_max = 380/365

print("GPU available :", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU :", torch.cuda.get_device_name(0))
    print("CUDA (PyTorch) :", torch.version.cuda)
    print("cuDNN :", torch.backends.cudnn.version())

!nvidia-smi

GPU available : True
GPU : Tesla V100-SXM2-32GB
CUDA (PyTorch) : 12.1
cuDNN : 8902
Tue Jan 13 14:01:44 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 575.51.03              Driver Version: 575.51.03      CUDA Version: 12.9     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla V100-SXM2-32GB           Off |   00000000:18:00.0 Off |                    0 |
| N/A   46C    P0             59W /  300W |    3920MiB /  32768MiB |      0%      Default |
|                                         |              

### DeepPricer network
This class implements the multilayer perceptron architecture used to train all Pricer models.

The network consists of four fully connected hidden layers with Softplus activations, ensuring smooth nonlinearities and stable gradients. A Softplus activation is applied at the output to enforce positivity of prices.

Using this exact class definition is essential, as it is required to correctly reload and reuse the trained DeepPricer models for inference and calibration.


In [3]:
import torch
import torch.nn as nn

class DeepPricer(nn.Module):
    def __init__(self, input_dim=5, hidden_layers=4, hidden_neurons=128, output_dim=1):
        """
        Neural network model for pricing, typically used to approximate option prices.

        Args:
            input_dim (int): Dimension of the input vector (default is 5 for SABR parameters).
            hidden_layers (int): Number of hidden layers in the network.
            hidden_neurons (int): Number of neurons per hidden layer.
            output_dim (int): Dimension of the output (default is 1 for option price).
        """
        super(DeepPricer, self).__init__()

        # Activation function used in all hidden layers: smooth approximation of ReLU
        self.activation = torch.nn.Softplus(beta=10., threshold=20.0)

        layers = []

        # First hidden layer: from input_dim to hidden_neurons
        layers.append(nn.Linear(input_dim, hidden_neurons))
        layers.append(self.activation)

        # Additional hidden layers: all with the same size
        for _ in range(hidden_layers - 1):
            layers.append(nn.Linear(hidden_neurons, hidden_neurons))
            layers.append(self.activation)

        # Output layer: maps the final hidden representation to the output
        layers.append(nn.Linear(hidden_neurons, output_dim))
        layers.append(nn.Softplus())

        # Combine all layers into a sequential module
        self.fc = nn.Sequential(*layers)

    def forward(self, params):
        """
        Forward pass through the network.

        Args:
            params (Tensor): Input tensor of shape (batch_size, input_dim)

        Returns:
            Tensor: Output tensor of shape (batch_size, output_dim)
        """
        return self.fc(params)


### Loading a trained DeepPricer
This utility function reloads a trained DeepPricer model from a saved checkpoint. It reconstructs the network architecture directly from the stored weights, without requiring manual specification of the dimensions.

The input dimension, number of hidden layers, hidden layer width, and output dimension are inferred from the shapes of the tensors in the state dictionary. This ensures consistency with the architecture used during training.

The function then instantiates the corresponding DeepPricer model and loads the learned parameters, making it possible to safely reuse trained pricers for inference.

In [4]:
import torch
import torch.nn as nn

def load_pricer_from_dir(dir_model):
    """
    Loads a DeepPricer model from a saved checkpoint, extracting the necessary dimensions and restoring weights.

    Args:
        dir_model (str): Path to the saved model checkpoint (.pt or .pth file).

    Returns:
        DeepPricer: An instance of the DeepPricer class with weights loaded from the checkpoint.
    """
    # Load checkpoint
    state_dict = torch.load(dir_model, map_location=torch.device('cpu'))

    # Infer dimensions from the state_dict
    all_keys = list(state_dict.keys())
    input_dim = state_dict[all_keys[0]].shape[1]
    output_dim = state_dict[all_keys[-2]].shape[0]  # Last linear layer's output size

    # Count number of hidden layers
    nb_linear_layers = sum('weight' in k and 'fc' in k for k in all_keys)
    hidden_layers = nb_linear_layers - 1  # remove output layer

    # Infer hidden neuron size from second linear layer (or first if only 1 hidden layer)
    hidden_neurons = state_dict[all_keys[2]].shape[0] if hidden_layers > 1 else state_dict[all_keys[0]].shape[0]

    # Instantiate model
    model = DeepPricer(
        input_dim=input_dim,
        hidden_layers=hidden_layers,
        hidden_neurons=hidden_neurons,
        output_dim=output_dim
    )

    # Load weights
    model.load_state_dict(state_dict)
    return model


### Model evaluation on validation data (prices and gradients)
This function evaluates a trained pricer on a validation CSV containing inputs, prices, and gradients.

The model first predicts prices $y_{\text{pred}} = f(x)$ and reports L1 errors, both in absolute value and as relative errors in percent. Very small targets are excluded from percentage metrics using a threshold $\varepsilon_{\text{price}}$ to avoid unstable ratios.

Gradients are then computed via autograd as $\nabla_\theta y_{\text{pred}}$ and compared to the reference gradients with the same L1 and relative (%) metrics, using per parameter masking with threshold $\varepsilon_{\text{grad}}$.

The output prints mean and standard deviation of errors for prices, and for each parameter gradient separately.


In [5]:
import torch
import pandas as pd

def evaluate_model(model, val_csv, n_param, device, max_samples=1_000_000, eps_price=0.01, eps_grad=0.01):
    """
    Evaluate a trained model on a validation set containing both prices and gradients,
    reporting L1 and relative (%) errors on both.

    Args:
        model (nn.Module): Trained model to evaluate.
        val_csv (str): Path to the test CSV file.
        n_param (int): Number of input parameters.
        device (torch.device): Computation device (CPU or CUDA).
        max_samples (int): Maximum number of samples to use for evaluation.
        eps_price (float): Threshold below which price values are ignored in % error.
        eps_grad (float): Threshold below which gradients are ignored in % error.
    """
    # Load data
    df = pd.read_csv(val_csv)
    print(f"📊 Testing dataset size: {len(df)}")
    if len(df) > max_samples:
        df = df.sample(n=max_samples, random_state=42).reset_index(drop=True)
        print(f"🔍 Subsampled to {max_samples} samples")

    # Convert to tensor
    tensor = torch.tensor(df.values, dtype=torch.float32, device=device)

    # Extract inputs, true prices, and true gradients
    x = tensor[:, :n_param].detach().clone().requires_grad_(True)
    y_true = tensor[:, n_param].unsqueeze(1)
    grad_true = tensor[:, n_param + 1:n_param + 1 + n_param]

    # Predict prices
    y_pred = model(x)

    # L1 price error
    l1_price_errors = torch.abs(y_pred - y_true)
    l1_price_mean = l1_price_errors.mean().item()
    l1_price_std = l1_price_errors.std().item()

    # Relative price error (%)
    rel_price_errors = 100.0 * l1_price_errors / (y_true.abs() + 1e-8)
    mask_price = y_true.abs().squeeze() > eps_price
    rel_price_filtered = rel_price_errors[mask_price]
    if rel_price_filtered.numel() > 0:
        rel_price_mean = rel_price_filtered.mean().item()
        rel_price_std = rel_price_filtered.std().item()
    else:
        rel_price_mean = float('nan')
        rel_price_std = float('nan')

    print(f"📈 L1 Price Error: mean = {l1_price_mean:.4e}, std = {l1_price_std:.4e}")
    print(f"📉 L1 Price Error (%): mean = {rel_price_mean:.4f}%, std = {rel_price_std:.4f}%")

    # Predict gradients via autograd
    grad_pred = torch.autograd.grad(
        y_pred, x,
        grad_outputs=torch.ones_like(y_pred),
        retain_graph=False,
        create_graph=False
    )[0].detach()

    # L1 gradient error
    l1_grad_errors = torch.abs(grad_pred - grad_true)
    l1_grad_mean = l1_grad_errors.mean(dim=0).cpu().numpy()
    l1_grad_std = l1_grad_errors.std(dim=0).cpu().numpy()

    # Relative gradient error (%), per parameter with masking
    rel_grad_errors = 100.0 * l1_grad_errors / (grad_true.abs() + 1e-8)
    rel_grad_mean = []
    rel_grad_std = []

    for j in range(n_param):
        mask_j = grad_true[:, j].abs() > eps_grad
        filtered = rel_grad_errors[mask_j, j]
        if filtered.numel() > 0:
            rel_grad_mean.append(filtered.mean().item())
            rel_grad_std.append(filtered.std().item())
        else:
            rel_grad_mean.append(float('nan'))
            rel_grad_std.append(float('nan'))
            print(f"⚠️ No significant gradients for parameter {j} (|grad| ≤ {eps_grad})")

    for i, (mean_g, std_g, mean_rel, std_rel) in enumerate(zip(l1_grad_mean, l1_grad_std, rel_grad_mean, rel_grad_std)):
        print(f"🧮 L1 Grad Error Param {i}: mean = {mean_g:.4e}, std = {std_g:.4e}")
        print(f"📊 L1 Grad Error Param {i} (%): mean = {mean_rel:.4f}%, std = {std_rel:.4f}%")


### Loading the four trained neural pricers
We load two DeepPricer models trained under different learning regimes, in order to compare the impact of price information and gradient information.

• **H1**: trained on prices and price gradients  
• **H2**: trained on prices only  

The two models share the same architecture and are loaded using the same class definition to ensure consistency across training regimes.


In [6]:
H1 = load_pricer_from_dir("utils/models/Heston_pricer(H1).pt").to(__device__)
H2 = load_pricer_from_dir("utils/models/Heston_pricer(H2).pt").to(__device__)

### Evaluation setup and interpretation
We now evaluate each neural pricer on a validation dataset containing 300,000 samples of prices and gradients. These reference quantities are computed using 1 million Monte Carlo paths.

Importantly, the targets are Monte Carlo prices and gradients generated by the GAN generator, not by the native model itself. This corresponds to the same generative model on which the pricers were trained, ensuring a consistent evaluation setting.

For each pricer, we report absolute L1 errors and relative percentage errors on both prices and parameter gradients. Gradient errors are displayed parameter by parameter, with the following ordering:

• Parameter 0: $\kappa$  
• Parameter 1: $\gamma$  
• Parameter 2: $\bar v$  
• Parameter 3: $\rho$  
• Parameter 4: $v_0$  
• Parameter 5: $T$  
• Parameter 6: $K$  

The outputs shown below, such as  
🧮 L1 Grad Error Param 0: mean = $9.40 \times 10^{-5}$, std = $1.02 \times 10^{-4}$  
📊 L1 Grad Error Param 0 (%): mean = $1.28\%$, std = $1.38\%$,  
should be interpreted as the average and dispersion of gradient approximation errors for each model parameter.


In [7]:
evaluate_model(H1, "utils/Sobolev_Test_Dataset/Heston_test_parameters.csv", n_param=7, device=__device__)

📊 Testing dataset size: 300000
📈 L1 Price Error: mean = 8.9721e-05, std = 8.5498e-05
📉 L1 Price Error (%): mean = 0.1445%, std = 0.1566%
🧮 L1 Grad Error Param 0: mean = 9.3976e-05, std = 1.0242e-04
📊 L1 Grad Error Param 0 (%): mean = 1.2778%, std = 1.3752%
🧮 L1 Grad Error Param 1: mean = 2.7387e-04, std = 2.6760e-04
📊 L1 Grad Error Param 1 (%): mean = 1.6100%, std = 1.6781%
🧮 L1 Grad Error Param 2: mean = 7.9959e-04, std = 8.0203e-04
📊 L1 Grad Error Param 2 (%): mean = 0.7096%, std = 1.3182%
🧮 L1 Grad Error Param 3: mean = 2.4877e-04, std = 2.7223e-04
📊 L1 Grad Error Param 3 (%): mean = 1.8175%, std = 1.9031%
🧮 L1 Grad Error Param 4: mean = 8.4219e-04, std = 9.0475e-04
📊 L1 Grad Error Param 4 (%): mean = 0.4706%, std = 1.1204%
🧮 L1 Grad Error Param 5: mean = 2.9496e-04, std = 2.9368e-04
📊 L1 Grad Error Param 5 (%): mean = 0.6388%, std = 0.8353%
🧮 L1 Grad Error Param 6: mean = 5.6132e-04, std = 5.2891e-04
📊 L1 Grad Error Param 6 (%): mean = 0.2079%, std = 0.5709%


In [8]:
evaluate_model(H2, "utils/Sobolev_Test_Dataset/Heston_test_parameters.csv", n_param=7, device=__device__)

📊 Testing dataset size: 300000
📈 L1 Price Error: mean = 1.3780e-04, std = 1.2249e-04
📉 L1 Price Error (%): mean = 0.2478%, std = 0.3478%
🧮 L1 Grad Error Param 0: mean = 1.9748e-04, std = 2.3865e-04
📊 L1 Grad Error Param 0 (%): mean = 3.0488%, std = 3.4336%
🧮 L1 Grad Error Param 1: mean = 7.5452e-04, std = 7.6508e-04
📊 L1 Grad Error Param 1 (%): mean = 4.3889%, std = 4.6137%
🧮 L1 Grad Error Param 2: mean = 3.8603e-03, std = 4.8217e-03
📊 L1 Grad Error Param 2 (%): mean = 3.4294%, std = 6.3446%
🧮 L1 Grad Error Param 3: mean = 6.9807e-04, std = 8.1560e-04
📊 L1 Grad Error Param 3 (%): mean = 5.2719%, std = 5.9845%
🧮 L1 Grad Error Param 4: mean = 4.4101e-03, std = 6.0281e-03
📊 L1 Grad Error Param 4 (%): mean = 2.3816%, std = 6.1764%
🧮 L1 Grad Error Param 5: mean = 9.2516e-04, std = 1.2739e-03
📊 L1 Grad Error Param 5 (%): mean = 1.9072%, std = 2.7678%
🧮 L1 Grad Error Param 6: mean = 1.7018e-03, std = 2.2293e-03
📊 L1 Grad Error Param 6 (%): mean = 0.6349%, std = 2.3365%
